# Data Preperation - revised based on Kyle Whynott’s CPSC 483 Intro to ML course

## 1. Data Set Selection

### Sources to Acquire Datasets From

- [Kaggle](https://www.kaggle.com/datasets)

- [UCI Machine Learning Repository](https://archive.ics.uci.edu/)

- [Data.Gov](https://data.gov/)

- [World Bank Open Data](https://data.worldbank.org/)

- [Awesome Public Datsets](https://github.com/awesomedata/awesome-public-datasets)

### Code Documentation

- [Numpy](https://numpy.org/)

- [Pandas](https://pandas.pydata.org/)

## 2. Import the Dataset

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# Show all columns in Jupyter Lab using Pandas
pd.set_option('display.max_columns', None)

In [ ]:
# Insert Dataset Here (Code to Be Completed)
# Option1 :
# if you use colab, you will have to mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
# your path is now "/content/drive/MyDrive/Colab Notebooks/dataset/air_quality/AirQualityUCI.csv"
df = pd.read_csv(
    "/content/drive/MyDrive/Colab Notebooks/dataset/air_quality/AirQualityUCI.csv",
    sep=";",
    decimal=","
)
# Option 2
# If you use your own IDE (VS Code, PyCharm, etc.),
# and your working directory is the project root,
# you can load the dataset using a relative path:
#df = pd.read_csv("./dataset/air_quality/AirQualityUCI.csv", sep=";",decimal=",")

In [ ]:
df.head()

## 3. Look for Data that Needs to Be Dropped / Cleaned

In [ ]:
# Drop Rows with NaN values ('how' indicates that all rows must contain NaN values to be dropped)
# Code to Be Completed Here
# this dataset uses -200 to indicate missing values
#load the dataset again just to show the difference
df = pd.read_csv("../dataset/air_quality/AirQualityUCI.csv", sep=";",decimal=",")
data_sz_b4_drop = len(df) #number of rows before cleaning

#To do list for cleaning the data
df = df.replace() #replacing -200 with NaN

df = df.dropna()   # drop empty columns with all “NaN / empty”
df = df.dropna() #drop rows with NaN values , by default axis=0, only one NaN value is enough to drop the row
# check how many rows left after cleaning


In [ ]:
display(df)

In [ ]:
#Too less data left after dropping rows, so we will try another method-> inserting median values instead of NaN
# Insert Dataset Here (Code to Be Completed)
#load the dataset again just to show the difference
df = pd.read_csv("../dataset/air_quality/AirQualityUCI.csv", sep=";",decimal=",")
#To do list for cleaning the data (keep more data this time)

df = df.replace() #replacing -200 with NaN
df = df.dropna()   # drop empty columns with all “NaN / empty”
df = df.fillna(df.median(numeric_only=True)) # fill-in median
after = len(df)

In [ ]:
display(df)

In [ ]:
target = "CO(GT)"
X2 = df.drop(columns=[target]) #X features
y2 = df[target] #y: label

In [ ]:
X2.shape  # (n_samples, n_features)
y2.shape  # (n_samples,)
print(X2.shape)
print(y2.shape)

## 4. Split the data into Training(70%)/Valitation(15%)/Testing(%15)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X2, y2,
    test_size=0.30,      # 30% left for val + test
    random_state=42
)
print(X_train.shape)  # (n_samples_train, n_features)
print(y_train.shape)  # (n_labels_train,)

In [ ]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,      # split 30% into 15% / 15%
    random_state=42
)
print(X_val.shape)  # (n_samples_train, n_features)
print(y_val.shape)  # (n_labels_train,)
print(X_test.shape)  # (n_samples_train, n_features)
print(y_test.shape)  # (n_labels_train,)

## 5. Now, let's use the built-in python code from the UCI database  
### Another method to load the datasset

In [ ]:
# Did it before, %pip install ucimlrepo, so skipping that step

In [ ]:
import ucimlrepo
print("ucimlrepo is installed")

In [ ]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
air_quality = fetch_ucirepo(id=360)

# data (as pandas dataframes)
X = air_quality.data.features
y = air_quality.data.targets

# metadata
print(air_quality.metadata)

# variable information
print(air_quality.variables)

In [ ]:
# print details of metadata (find indication of missing values)

from pprint import pprint
pprint(air_quality.metadata)

In [ ]:
X = air_quality.data.features

In [ ]:
print(X.shape)

In [ ]:
display(X.head(10))#show first 10 rows

In [ ]:
#y is currently empty dataframe,
display(y)
#to move the CO from features to target
y = X["CO(GT)"]
X = X.drop(columns=["CO(GT)"])

In [ ]:
#Show more details about the data
type(X)
X.info()

In [ ]:
# Describe the data (include some statistics)
#As you can see there are lots of "-200" values in the dataset, they are invalid
X.describe()

In [ ]:
#check for missing values
X.isna().sum()

In [ ]:
#check first column values
X.iloc[:, 0]

In [ ]:
#check data type of first column
#type(X.iloc[:, 0])
#check data type of 4th column
type(X.iloc[:, 3])
display(X.iloc[0:10, 3])
display(X.head(10))

### You can use similar way to split the training/validation/testing data
### Please be aware that -200 is not removed in X and y (using the direct method to loaded in UCI Air Quality dataset)
### In the next section, we still use the "df" datafram we originally loaded from *CSV file (invalid -200 is replaced by median)  
--------------------------------------------------------------------------------------------------------------------------------------

## CO(GT) Regression Demo

This section shows two regressions to predict **CO(GT)**:

1. **Baseline (1 feature, not a sensor):** use only **NOx(GT)**
2. **Improved (multiple meaningful features, no CO sensor ):** use **NOx(GT), NO2(GT), NMHC(GT), T, RH, AH**

> Note: We intentionally **exclude** `PT08.S1(CO)` (the CO sensor) in the improved model to avoid leakage.


In [ ]:
# Ensure the cleaned dataframe `df` exists
import pandas as pd
import numpy as np

required_cols = ["CO(GT)", "NOx(GT)", "NO2(GT)", "NMHC(GT)", "T", "RH", "AH"]

# Keep only rows with the columns we need
reg_df = df[required_cols].dropna().copy()
print("Regression dataset shape:", reg_df.shape)
reg_df.head()


### 1) Baseline: predict CO(GT) using **one** feature (NOx(GT))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np
# scikit-learn is a Python library that provides ready-to-use machine learning algorithms
# for documentation of scikit-learn, please check https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html

X_base = reg_df[["NOx(GT)"]] #we didn't do normalization or standardization because only "one" feature, and linear regression is not sensitive to feature scaling
y = reg_df["CO(GT)"]

#Because no model selection, we only slit the data into train and test sets, no validation set needed
X_train, X_test, y_train, y_test = train_test_split(
    X_base, y, test_size=0.2, random_state=42
)

baseline_model = LinearRegression()#
baseline_model.fit(X_train, y_train)

y_pred_base = baseline_model.predict(X_test)

print("Baseline (1 feature = NOx(GT))")

print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_base)))
#Note:RMSE tells you how big the errors are; R² tells you how useful the model is compared to always predicting the mean of y_train.

#### To do: print R^2 : it means how well your model explains the variability in the data
####----------------------------------------------------------------------------------------

In [ ]:
# Optional: quick visualization (actual vs predicted)
import matplotlib.pyplot as plt

plt.figure()
plt.scatter(y_test, y_pred_base)
plt.xlabel("Actual CO(GT)")
plt.ylabel("Predicted CO(GT)")
plt.title("Baseline: CO(GT) from NOx(GT) only")
plt.show()


In [ ]:
#Baseline: Predicted vs True plot (one feature)
import matplotlib.pyplot as plt

plt.figure(figsize=(5, 5))

# scatter of predictions vs truth
plt.scatter(y_test, y_pred_base, alpha=0.5) #alpha = 0.5 → 50% transparent

# perfect prediction line
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    "r--"
)

plt.xlabel("True CO(GT)")
plt.ylabel("Predicted CO(GT)")
plt.title("Baseline model: Prediction vs Truth (1 feature)")
plt.show()

#If predictions were perfect, all points would lie on the red diagonal.

In [ ]:
#Baseline: one feature → regression line
import matplotlib.pyplot as plt
import numpy as np

# Sort values for a clean line
idx = np.argsort(X_test.values.flatten())#If you don’t sort, the red “regression line” will look messy or zig-zaggy, even though the model is correct

plt.figure(figsize=(6, 4))
plt.scatter(X_test.values, y_test, alpha=0.5, label="True data")
plt.plot(
    X_test.values[idx],
    y_pred_base[idx],
    color="red",
    linewidth=2,
    label="Regression line"
)

plt.xlabel("NOx(GT)")
plt.ylabel("CO(GT)")
plt.title("Baseline: CO vs NOx")
plt.legend()
plt.show()


### 2) Improved: predict CO(GT) using multiple meaningful features (no CO sensor)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

features = ["NOx(GT)", "NO2(GT)", "NMHC(GT)", "T", "RH", "AH"]
X = reg_df[features]
y = reg_df["CO(GT)"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Let's use Ridge model:  Ridge helps when features are correlated (common in air quality data)
# Pipeline is a class provided by the scikit-learn library that allows you to chain together multiple steps in a machine learning workflow, such as preprocessing and modeling, into a single object.
# This makes it easier to manage and apply the same transformations to both training and testing data.

model = Pipeline([
    ("scaler", StandardScaler()), #normalization.
    ("ridge", Ridge(alpha=1.0))
])

#Note : if you don't want to combine two steps, you can do them separately like this:
#scaler = StandardScaler()
#ridge = Ridge(alpha=1.0)
#For training
#X_train_scaled = scaler.fit_transform(X_train)
#ridge.fit(X_train_scaled, y_train)
#for prediction
#ridge.predict(scaler.transform(X_test))

#feed the training data to the model, fit means to train the model using the training data (parameters will be learned during the training process)
model.fit(X_train, y_train)
#Let's use the test data to evaluate the model
y_pred = model.predict(X_test)

print("Improved (multiple features)")
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred))) #On average, predictions are off by about RMSE units of CO(GT).
#Note:RMSE tells you how big the errors are; R² tells you how useful the model is compared to always predicting the mean of y_train.

#### To do: print R^2 : it means how well your model explains the variability in the data
####-------------------------------------------------------------------------------

In [ ]:
# Optional: visualization (actual vs predicted)
plt.figure()
plt.scatter(y_test, y_pred)
plt.xlabel("Actual CO(GT)")
plt.ylabel("Predicted CO(GT)")
plt.title("Improved: CO(GT) from NOx/NO2/NMHC + Weather")
plt.show()


In [ ]:
#How to visualize a multi-feature regression model’s predictions vs true values?
#Option A : Predicted vs True plot
#“If predictions were perfect, all points would lie on the red diagonal.”

plt.figure(figsize=(5, 5))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    "r--"
)

plt.xlabel("True CO(GT)")
plt.ylabel("Predicted CO(GT)")
plt.title("Multi-feature model: Prediction vs Truth")
plt.show()


In [ ]:
#side by side comparison of baseline vs improved
import matplotlib.pyplot as plt

# Create 1 row, 2 columns of subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1: Baseline model
ax1.scatter(y_test, y_pred_base, alpha=0.5)
ax1.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
ax1.set_xlabel("True CO(GT)")
ax1.set_ylabel("Predicted CO(GT)")
ax1.set_title("Baseline Model")

# Plot 2: Improved model
ax2.scatter(y_test, y_pred, alpha=0.5)
ax2.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
ax2.set_xlabel("True CO(GT)")
ax2.set_ylabel("Predicted CO(GT)")
ax2.set_title("Improved Model")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error
# Display metrics side by side
rmse_baseline = np.sqrt(mean_squared_error(y_test, y_pred_base))
rmse_improved = np.sqrt(mean_squared_error(y_test, y_pred))
r2_baseline = r2_score(y_test, y_pred_base)
r2_improved = r2_score(y_test, y_pred)
#### To do: add R^2 for comparison : it means how well your model explains the variability in the data

print("=" * 50)
print("RMSE Comparison")
print("=" * 50)
print(f"Baseline Model RMSE:  {rmse_baseline:.4f}")
print(f"Improved Model RMSE:  {rmse_improved:.4f}")
print(f"Improvement:          {((rmse_baseline - rmse_improved) / rmse_baseline * 100):.2f}%")
print("=" * 50)


print("=" * 50)
print("R^2 Comparison??????? please complete")
print("=" * 50)


### (HW2)  Part I
### 3) Select another candidate model to predict CO(GT) using multiple meaningful features (no CO sensor)
#### Please use similar method shown in 2) but try different model for example Lasso Regression (https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html)

#### Please make sure you include the following parts (they can be found in (2))
#### 1. Predicted vs True plot (red diagonal line is displayed as a ideal prediction)
#### 2. Side by side comparison (plots) of the model used in (2) vs the model used in (3)-> this HW Part I section
#### 3. Display metrics side by side


In [ ]:
# please add your HW Part I cells here

### (HW2)  Part II
### 4) Based on previous model evaluaiton metric, you can pick one model with better performance
### Adjust the model parameters, ex. alpha value in Ridge model, then predict CO(GT) using multiple meaningful features (no CO sensor)
#### Please make sure you include the following parts (they can be found in (2))
#### 1. Predicted vs True plot (red diagonal line is displayed as a ideal prediction)
#### 2. Side by side comparison (plots) of the model you pick with the original parameter(s) and with the adjusted parameter(s)
#### 3. Display metrics side by side


In [ ]:
# please add your HW Part II here

(HW2) Part III

### Discussion: Model and Parameter Selection

Based on your evaluations from Part I and Part II, write a concise discussion addressing the following points:

1.  **Compare Model Performance:** How did the Lasso Regression model (from HW Part I, or any other model you selected) perform compared to the Ridge Regression model in predicting CO(GT)? Refer to RMSE and R^2 metrics.
2.  **Impact of Parameter Tuning:** How did adjusting the parameter in the Ridge/Lasso model affect its performance (as practived in HW Part II)? Provide evidence from your metrics.
3.  **Best Performing Model/Parameter:** Which model (and with which parameters) did you find to be the most effective for predicting CO(GT) and why? Justify your choice based on the evaluation metrics.



#### Please type your disussion for HW part III here ---> Below